# Person A — Notebook 1 FAST (GPT-2-XL): ACDC Circuit Discovery
**CS 590NN | Amogh | Apr 19 — FAST GPT-2-XL arm**

Fast version of the GPT-2-XL circuit discovery. Reduces scope to keep runtime
tractable on top of Qwen arm.

**Speed config:**
- **N_SAMPLES = 20** (was 50)
- **threshold = 0.4** (tighter than 0.3 → fewer attention patches survive)
- Everything else identical to the v2.1 ACDC implementation

**Output:** `week2_circuit_log_gpt2xl.json`

**Runtime:** ~15-20 min on H100.


### 1.0 Install

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)
pip(["numpy==1.26.4"])
pip(["transformer-lens", "transformers", "datasets", "accelerate", "einops", "jaxtyping"])
print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)


### 1.1 Imports — GPU check upfront

In [ ]:
import torch, json, re, requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer

# Fail fast if GPU is missing — avoids wasting 1 min downloading 6 GB to CPU
assert torch.cuda.is_available(), (
    "CUDA not available. Go to Runtime → Change runtime type → Hardware accelerator → "
    "GPU (H100 if possible). Then Disconnect → Reconnect and re-run."
)

DEVICE = "cuda"
print(f"GPU    : {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info()
print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2, 0)

torch.manual_seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Imports OK")


### 1.2 Load GPT-2-XL

In [ ]:
MODEL_NAME = "gpt2-xl"
print(f"Loading {MODEL_NAME}... (~1 min)")
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME} | {model.cfg.n_layers} layers | {model.cfg.n_heads} heads")
print(f"GPU free after load: {free/1e9:.1f} GB")


### 1.3 ACDC (threshold=0.4, effect_floor=0.5)

In [ ]:
class NativeACDC:
    """ACDC via activation patching. Threshold tuned for GPT-2-XL."""
    def __init__(self, model, threshold=0.4, effect_floor=0.5):
        self.model = model
        self.threshold = threshold
        self.effect_floor = effect_floor

    def _logit_diff(self, logits, target_id, baseline_id):
        return (logits[0, -1, target_id] - logits[0, -1, baseline_id]).item()

    def run(self, prompt, target_true, target_new):
        true_id = self.model.to_tokens(target_true, prepend_bos=False)[0, 0].item()
        new_id  = self.model.to_tokens(target_new,  prepend_bos=False)[0, 0].item()
        tokens  = self.model.to_tokens(prompt)

        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, self.model.cfg.d_vocab-1, (tokens.shape[1]-2,))

        with torch.no_grad():
            clean_logits,   clean_cache   = self.model.run_with_cache(tokens)
            corrupt_logits, corrupt_cache = self.model.run_with_cache(corrupt)

        clean_score   = self._logit_diff(clean_logits,   true_id, new_id)
        corrupt_score = self._logit_diff(corrupt_logits, true_id, new_id)
        raw_effect    = abs(clean_score - corrupt_score)
        effect_range  = max(raw_effect, self.effect_floor)
        effect_clamped = raw_effect < self.effect_floor

        causal_scores = {}
        for layer in range(self.model.cfg.n_layers):
            hn = f"blocks.{layer}.attn.hook_result"
            if hn in clean_cache:
                for head in range(self.model.cfg.n_heads):
                    def make_attn(h=head, n=hn):
                        ca = corrupt_cache[n][:, :, h:h+1, :].clone()
                        def fn(v, hook): v[:, :, h:h+1, :] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_attn())])
                    causal_scores[f"attn_{layer}_{head}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

            hn = f"blocks.{layer}.hook_mlp_out"
            if hn in clean_cache:
                def make_mlp(n=hn):
                    ca = corrupt_cache[n].clone()
                    def fn(v, hook): return ca
                    return fn
                with torch.no_grad():
                    pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_mlp())])
                causal_scores[f"mlp_{layer}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

        del clean_cache, corrupt_cache, clean_logits, corrupt_logits
        torch.cuda.empty_cache()

        attn_heads, mlp_layers = [], []
        for node, score in causal_scores.items():
            if score >= self.threshold:
                parts = node.split("_")
                if parts[0] == "attn": attn_heads.append((int(parts[1]), int(parts[2])))
                else:                  mlp_layers.append(int(parts[1]))

        return {
            "attn_heads":     sorted(set(attn_heads)),
            "mlp_layers":     sorted(set(mlp_layers)),
            "clean_score":    clean_score,
            "corrupt_score":  corrupt_score,
            "raw_effect":     raw_effect,
            "effect_clamped": effect_clamped,
        }

print("NativeACDC defined.")


### 1.4 Load CounterFact (20 samples)

In [ ]:
@dataclass
class EditSample:
    prompt: str; target_new: str; target_true: str
    related_prompts: List[str] = field(default_factory=list)

raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
N_SAMPLES = 20

def parse_counterfact(raw):
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" " + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=[p for p in raw.get("neighborhood_prompts", [])[:3] if isinstance(p, str)],
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples (fast subset of the 50 used on Qwen)")


### 1.5 Run ACDC on 20 Samples

In [ ]:
acdc = NativeACDC(model, threshold=0.4, effect_floor=0.5)
circuit_log = []

print(f"Running ACDC on GPT-2-XL — {model.cfg.n_layers} layers × {model.cfg.n_heads} heads")
print(f"  threshold={acdc.threshold}, effect_floor={acdc.effect_floor}\n")

for i, s in enumerate(cf_samples):
    result = acdc.run(s.prompt, s.target_true, s.target_new)
    circuit_log.append({
        "idx": i, "prompt": s.prompt,
        "attn_heads": result["attn_heads"], "mlp_layers": result["mlp_layers"],
        "clean_score": round(result["clean_score"], 4),
        "corrupt_score": round(result["corrupt_score"], 4),
        "raw_effect": round(result["raw_effect"], 4),
        "effect_clamped": bool(result["effect_clamped"]),
        "n_attn": len(result["attn_heads"]),
        "n_mlp":  len(result["mlp_layers"]),
    })
    if i % 3 == 0:
        free = torch.cuda.mem_get_info()[0] / 1e9
        clamp = " [CLAMPED]" if result["effect_clamped"] else ""
        print(f"  [{i+1:>2}/{N_SAMPLES}]  n_attn={len(result['attn_heads']):>3}  "
              f"n_mlp={len(result['mlp_layers']):>2}  raw_eff={result['raw_effect']:.2f}{clamp}  gpu_free={free:.1f}GB")

with open(RESULTS_DIR / "week2_circuit_log_gpt2xl.json", "w") as f:
    json.dump(circuit_log, f, indent=2)

from collections import Counter
all_attn = [str(h) for e in circuit_log for h in e["attn_heads"]]
all_mlp  = [str(l) for e in circuit_log for l in e["mlp_layers"]]
print(f"\nAvg n_attn = {sum(e['n_attn'] for e in circuit_log)/N_SAMPLES:.2f}")
print(f"Avg n_mlp  = {sum(e['n_mlp']  for e in circuit_log)/N_SAMPLES:.2f}")
print(f"Top heads  : {Counter(all_attn).most_common(5)}")
print(f"Top MLPs   : {Counter(all_mlp).most_common(5)}")
print(f"Saved -> {RESULTS_DIR}/week2_circuit_log_gpt2xl.json")


### 1.6 Save + Download

In [ ]:
import shutil
from datetime import datetime
from collections import Counter

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
n_attn_list = [e["n_attn"] for e in circuit_log]
n_mlp_list  = [e["n_mlp"]  for e in circuit_log]
n_clamped   = sum(1 for e in circuit_log if e["effect_clamped"])

QWEN_V21_AVG_ATTN = 21.4
QWEN_V21_AVG_MLP  = 10.96

summary = {
    "author": "Amogh",
    "notebook": "Notebook 1 FAST (GPT-2-XL) — ACDC Circuit Discovery",
    "model": MODEL_NAME, "timestamp": timestamp,
    "n_samples": N_SAMPLES,
    "threshold": acdc.threshold, "effect_floor": acdc.effect_floor,
    "fast_version": True,
    "circuit_summary": {
        "avg_attn_per_sample": round(sum(n_attn_list) / N_SAMPLES, 2),
        "avg_mlp_per_sample":  round(sum(n_mlp_list) / N_SAMPLES, 2),
        "median_n_attn": sorted(n_attn_list)[N_SAMPLES//2],
        "median_n_mlp":  sorted(n_mlp_list)[N_SAMPLES//2],
        "max_n_attn": max(n_attn_list), "max_n_mlp": max(n_mlp_list),
        "top_attn_heads": Counter(all_attn).most_common(5),
        "top_mlp_layers": Counter(all_mlp).most_common(5),
        "n_samples_effect_clamped": n_clamped,
    },
    "comparison_vs_qwen_v21": {
        "avg_attn": {"qwen": QWEN_V21_AVG_ATTN, "gpt2xl": round(sum(n_attn_list)/N_SAMPLES, 2)},
        "avg_mlp":  {"qwen": QWEN_V21_AVG_MLP,  "gpt2xl": round(sum(n_mlp_list)/N_SAMPLES, 2)},
    }
}

with open(RESULTS_DIR / f"summary_nb1_gpt2xl_{timestamp}.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = f"/content/PersonA_Notebook1_gpt2xl_FAST_{timestamp}"
shutil.make_archive(zip_path, "zip", RESULTS_DIR)

print("=" * 66)
print(f"  NB1 FAST (GPT-2-XL) — Amogh  ({timestamp})")
print("=" * 66)
print(f"  Model: {MODEL_NAME} | {model.cfg.n_layers}L × {model.cfg.n_heads}H")
print()
print(f"  {'metric':<20} {'Qwen 0.6B v2.1':>16} {'GPT-2-XL':>12}")
print(f"  {'avg n_attn':<20} {QWEN_V21_AVG_ATTN:>16.2f} {summary['circuit_summary']['avg_attn_per_sample']:>12.2f}")
print(f"  {'avg n_mlp':<20} {QWEN_V21_AVG_MLP:>16.2f} {summary['circuit_summary']['avg_mlp_per_sample']:>12.2f}")
print(f"\n  Effect-clamped: {n_clamped}/{N_SAMPLES}")
print(f"  Download: {zip_path}.zip")

from google.colab import files
files.download(f"{zip_path}.zip")
